# 📅 Enterprise Date Dimension (`dim_date`) Creation
This notebook builds the conformed calendar dimension table `fmcg.gold.dim_date` at monthly grain, providing standardized time attributes (fiscal years, quarters, month names, and numeric surrogate date keys) for enterprise analytics.

### 📌 Step 1: Import Dependencies & Load Lakehouse Utilities
* **Purpose:** Imports PySpark SQL functions and executes the shared Lakehouse utilities to obtain configuration, schemas, and catalog references.
* **Logic & Transformations:** Imports `pyspark.sql.functions as F` and runs `%run ./utilities` to establish shared catalog and schema variables.
* **Inputs & Dependencies:** Shared utilities module (`./utilities.py` / `utilities.ipynb`).
* **Outputs & Medallion State:** Active Spark session with centralized `catalog`, `bronze_schema`, `silver_schema`, and `gold_schema` in scope.

In [1]:
from pyspark.sql import functions as F
%run ./utilities


### 📌 Step 2: Parameterize Date Dimension Boundaries via Widgets
* **Purpose:** Establishes configurable start and end dates for calendar sequence generation, allowing flexible backfilling or forward projection.
* **Logic & Transformations:** Registers interactive Databricks text widgets `start_date` (default: `2024-01-01`) and `end_date` (default: `2026-12-01`).
* **Inputs & Dependencies:** Databricks widget inputs.
* **Outputs & Medallion State:** Variables `start_date` and `end_date` populated for sequence generation.

In [2]:
# Parameterize date range with sensible defaults
dbutils.widgets.text("start_date", "2024-01-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2026-12-01", "End Date (YYYY-MM-DD)")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")


### 📌 Step 3: Generate Monthly Sequence & Enrich Calendar Attributes
* **Purpose:** Synthesizes a contiguous monthly calendar sequence and enriches each month with corporate reporting attributes.
* **Logic & Transformations:**
  1. Generates month-start dates using `sequence(to_date(start_date), to_date(end_date), interval 1 month)` exploded into `month_start_date`.
  2. Derives integer surrogate key `date_key` (`yyyyMM`, e.g. `202401`).
  3. Extracts calendar attributes: `year`, `month_name` (`January`), `month_short_name` (`Jan`), `quarter` (`Q1`), and `year_quarter` (`2024-Q1`).
* **Inputs & Dependencies:** `start_date` and `end_date` parameters.
* **Outputs & Medallion State:** Enriched DataFrame `df` containing contiguous monthly date dimension attributes.

In [3]:
# 1️⃣ Generate one row per month start between start_date and end_date
df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 month
            )
        ) AS month_start_date
    """)
)


# 2️⃣ Add useful analytics columns
df = (
    df
    # Surrogate key at month grain
    .withColumn("date_key", F.date_format("month_start_date", "yyyyMM").cast("int"))
    .withColumn("year", F.year("month_start_date"))
    .withColumn("month_name", F.date_format("month_start_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("month_start_date", "MMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("month_start_date")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.quarter("month_start_date")))
)

### 📌 Step 4: Preview Generated Date Dimension Data
* **Purpose:** Inspects generated calendar records to verify attribute formatting, sorting, and surrogate key alignment before persistence.
* **Logic & Transformations:** Calls `display(df)` to render top 20 rows of the calendar dataset in tabular format.
* **Inputs & Dependencies:** Enriched DataFrame `df` from Step 3.
* **Outputs & Medallion State:** Visual tabular preview rendered in notebook output.

In [4]:
display(df)

+----------------+--------+----+----------+----------------+-------+------------+
|month_start_date|date_key|year|month_name|month_short_name|quarter|year_quarter|
+----------------+--------+----+----------+----------------+-------+------------+
|2024-01-01      |202401  |2024|January   |Jan             |Q1     |2024-Q1     |
|2024-02-01      |202402  |2024|February  |Feb             |Q1     |2024-Q1     |
|2024-03-01      |202403  |2024|March     |Mar             |Q1     |2024-Q1     |
|2024-04-01      |202404  |2024|April     |Apr             |Q2     |2024-Q2     |
|2024-05-01      |202405  |2024|May       |May             |Q2     |2024-Q2     |
|2024-06-01      |202406  |2024|June      |Jun             |Q2     |2024-Q2     |
|2024-07-01      |202407  |2024|July      |Jul             |Q3     |2024-Q3     |
|2024-08-01      |202408  |2024|August    |Aug             |Q3     |2024-Q3     |
|2024-09-01      |202409  |2024|September |Sep             |Q3     |2024-Q3     |
|2024-10-01     

### 📌 Step 5: Persist Conformed Date Dimension to Gold Delta Lake
* **Purpose:** Writes the calendar dataset to the enterprise gold layer table `fmcg.gold.dim_date` using Delta Lake format.
* **Logic & Transformations:** Writes `df` in `overwrite` mode as a managed Delta table under `{catalog}.{gold_schema}.dim_date`.
* **Inputs & Dependencies:** Enriched DataFrame `df` and target table reference `target_table`.
* **Outputs & Medallion State:** Delta table `{catalog}.gold.dim_date` persisted and ready for star-schema joins with fact tables.

In [5]:
# Save as Delta table under configured catalog and schema
target_table = f"{catalog}.{gold_schema}.dim_date"
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(target_table)
print(f"Successfully wrote date dimension to {target_table}")


[Local Spark Emulation] Adapted target table: fmcg.gold.dim_date -> gold.dim_date


26/09/17 11:27:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Successfully wrote date dimension to fmcg.gold.dim_date
